# Issue #206 — Adaptive Timeboxing Session Kernel

- Status: WIP
- Owner: Hugo + coding agent
- GitHub Issue: https://github.com/hugocool/FateForger/issues/206
- Branch: `issue/206-adaptive-timeboxing-stage-contract`
- GitHub PR: none until branch publication
- Acceptance criteria: AC1–AC7 from issue #206
- Last clean run: none yet; project `.venv`; Python 3.13.7
- Worktree baseline: clean at `fd95a0e5cdabe0a5f65c812830b515cbd902ff41` on 2026-08-29T15:56:57+0200; `git status --porcelain` returned no entries
- Original worktree baseline: dirty with unrelated user files; those files are absent from this isolated worktree and remain untouched


## Pairing Intake Record

**Confirmed problem.** The timeboxing agent treated ordinary planner-owned placement choices as unanswered user questions, lost authority over the planning date/day type, and could advance without producing the next reviewable artifact. Restart continuity and instruction contamination made the behavior worse.

**Constraints.** Slack Block Kit cards are the user surface. The host locks the planning date and derives weekday/day type. Stage 3 presents a skeleton only; Stage 4 performs the first tmbx patch. NL and UI actions converge on typed intents. No regex/keyword NLU. No private reasoning is surfaced. Progress is observational. Persistence is SQLAlchemy/Alembic.

**Ownership.** The user owns goals, immovable commitments, genuine preferences, and trade-offs. The planner owns ordinary placement and sequencing assumptions. The host owns date derivation, persistence, artifact gates, external context, and exact candidate commit identity.

**Selected direction.** Artifact-led adaptive session kernel, approved by Hugo on 2026-08-29.

**Open questions.** None about design. Hugo acknowledged continuous implementation on 2026-08-29 and waived intermittent approvals. Commits/pushes remain deferred until final handoff unless explicitly authorized.


## Design Options

1. **Artifact-led kernel — selected.** Derive the next action from durable typed artifacts and approvals. Best fit for adaptive planning, replay, and restart safety. Main risk: a deliberate migration seam from the transcript-led handler.
2. **Slack streaming orchestrator.** Keep the current harness-centric flow and add richer stage/progress events around it. Smaller surface change, but it preserves transcript state and cannot reliably enforce artifact advancement.
3. **Event-sourced kernel.** Store every intent/artifact transition as an append-only domain event. Strongest auditability, but substantially more infrastructure and projection complexity than #206 needs.

Approved specification: [adaptive timeboxing session kernel design](../../docs/superpowers/specs/2026-08-29-adaptive-timeboxing-session-kernel-design.md).


## Selected Direction and Pseudocode

```text
turn(request, progress):
  snapshot = repository.load_or_create(session_key, owner)
  return stored outcome if interaction was already handled
  reject stale revision or unauthorized actor
  snapshot = apply_typed_intent(snapshot, request.intent)
  enforce planning-day and pending-approval gates
  target = derive_next_artifact(snapshot)
  readiness = requirements.evaluate(target, snapshot)
  if readiness has a hard user-owned blocker: ask one reasoned question
  context = host.fetch_required_context(readiness)
  result = planner.produce(complete_planning_brief(snapshot, context, target))
  reject user blockers for planner-owned requirements or a missing target artifact
  persist snapshot and outcome atomically
  return the next reviewable artifact or commit receipt
```

Progress emissions go through a separate best-effort sink and never determine domain success.


## AC-to-Artifact Mapping

| AC | Observable artifact/evidence |
|---|---|
| AC1 locked date/day type | `PlanningDay`, date-card metadata, SQL round trip, Saturday/weekend replay |
| AC2 adaptive downstream readiness | requirement catalog + ownership report + blocker tests |
| AC3 planner autonomy | planner-owned assumption tests and no unnecessary gym/start-time question |
| AC4 typed advance/artifact guarantee | discriminated intents/outcomes, typed DeepSeek result, Stage 3 skeleton outcome |
| AC5 restart continuity | Alembic-backed snapshot and new-repository replay |
| AC6 instruction hygiene | complete host brief, profile assertions, no transcript contamination |
| AC7 exact replay | deterministic incident fixture plus live Slack/log/metric audit |


## Implementation Walkthrough / Decision Audit

Implementation proceeds AC-by-AC under TDD. Each slice records the alternative considered, selected path, rationale, files, test command, observed RED, and observed GREEN.

**Task 2 — typed session contracts.** Selected strict Pydantic models with canonical artifact identity instead of loose dictionaries. RED was the missing module. GREEN after review fixes: 10 contract tests. Independent review found and caused fixes for calendar-basis classification forgery, shaped-but-incorrect artifact digests, and duplicate fact IDs at the intent boundary.

**Task 3 — downstream readiness and ownership policy.** Selected a typed requirement catalog and transitive artifact dependency graph instead of stage mutation or free-form message parsing. The incident readiness cell below verifies that an ordinary gym with no exact time is a planner-owned `assume` gap, while the skeleton has no hard user blocker. RED was the missing module; GREEN was 8 readiness tests, with 18 Task 2+3 contract/readiness tests passing.

**Task 4 — in-memory adaptive kernel.** Selected one deep artifact-led `turn()` facade with typed ports and repository-backed replay instead of stage authority or transcript reconstruction. The executable evidence cell below imports the extracted implementation and proves that one `Advance` produces the Stage 3 skeleton with a labeled planner-owned gym assumption, exactly one planner call, and no commit call. Strict TDD plus two review rounds ended at 19 focused kernel tests and 37 cumulative Task 2–4 tests.


## Executable Walkthrough

These initial cells verify the approved design artifact and pin the sanitized incident identity before production code exists. Later cells will import extracted modules; no `sys.path` manipulation is allowed.


In [ ]:
from pathlib import Path

SPEC = Path("../../docs/superpowers/specs/2026-08-29-adaptive-timeboxing-session-kernel-design.md")
assert SPEC.exists()
print(SPEC)


In [ ]:
INCIDENT = {
    "session_key": "C0AA6HC1RJL:1787995886.748859",
    "planning_date": "2026-08-29",
    "timezone": "Europe/Amsterdam",
    "expected_iso_weekday": 6,
    "expected_day_type": "weekend",
}
INCIDENT


In [ ]:
from datetime import date

from fateforger.agents.timeboxing.session_contracts import DayType, PlanningDay

locked_day = PlanningDay.lock_default(
    value=date.fromisoformat(INCIDENT["planning_date"]),
    timezone=INCIDENT["timezone"],
    lock_revision=1,
)
assert locked_day.iso_weekday == INCIDENT["expected_iso_weekday"]
assert locked_day.day_type is DayType.WEEKEND
locked_day.model_dump(mode="json")


In [ ]:
from fateforger.agents.timeboxing.readiness import (
    RequirementOwner,
    TimeboxRequirements,
)
from fateforger.agents.timeboxing.session_contracts import (
    ArtifactKind,
    FactKind,
    PlanningFact,
    PlanningSessionSnapshot,
)

incident_readiness_snapshot = PlanningSessionSnapshot(
    session_key=INCIDENT["session_key"],
    revision=1,
    owner_user_id="U-sanitized",
    planning_day=locked_day,
    facts=[
        PlanningFact(
            fact_id="activity-1",
            kind=FactKind.REQUESTED_ACTIVITY,
            value="Prepare the presentation",
            source="user",
            source_interaction_id="interaction-sanitized",
        ),
        PlanningFact(
            fact_id="gym-1",
            kind=FactKind.GYM,
            value=True,
            source="user",
            source_interaction_id="interaction-sanitized",
        ),
    ],
)

incident_readiness = TimeboxRequirements().evaluate(
    ArtifactKind.SKELETON, incident_readiness_snapshot
)
gym_placement = incident_readiness.by_id("skeleton.gym_placement")
assert incident_readiness.first_hard_user_blocker() is None
assert gym_placement.owner is RequirementOwner.PLANNER
assert gym_placement.resolution == "assume"
incident_readiness


In [ ]:
from fateforger.agents.timeboxing.adaptive_timeboxing import (
    AdaptiveTimeboxing,
    InMemoryPlanningSessionRepository,
    PlanningContext,
    TurnRequest,
)
from fateforger.agents.timeboxing.session_contracts import (
    Advance,
    ArtifactDraft,
    AwaitingApproval,
    PlannerAssumptionDraft,
    PlanningArtifact,
    PlanningResult,
)

class NotebookPlanner:
    def __init__(self):
        self.calls = 0

    async def produce(self, brief, progress):
        self.calls += 1
        return PlanningResult(
            artifact_updates=[
                ArtifactDraft(
                    kind=ArtifactKind.SKELETON,
                    payload={"markdown": "## Saturday\n- 17:00 Gym"},
                    dependency_revisions={"planning_day": 1},
                )
            ],
            assumptions=[
                PlannerAssumptionDraft(
                    requirement_id="skeleton.gym_placement",
                    value="17:00",
                    why_needed="place gym around the dinner anchor",
                    invalidated_by=["gym", "dinner"],
                )
            ],
        )

class NotebookContext:
    async def propose_planning_day(self, request):
        return locked_day

    async def resolve(self, snapshot, *, target, progress):
        return PlanningContext(
            applicable_constraints={"items": []},
            calendar_snapshot={"events": []},
        )

class NotebookCommit:
    def __init__(self):
        self.calls = 0

    async def commit(
        self, candidate: PlanningArtifact, *, digest: str
    ) -> PlanningArtifact:
        self.calls += 1
        raise AssertionError("Stage 3 must not commit")

class NotebookProgress:
    async def emit(self, event):
        pass

task4_repo = InMemoryPlanningSessionRepository([incident_readiness_snapshot])
task4_planner = NotebookPlanner()
task4_commit = NotebookCommit()
task4_kernel = AdaptiveTimeboxing(
    repository=task4_repo,
    requirements=TimeboxRequirements(),
    planner=task4_planner,
    context=NotebookContext(),
    commit=task4_commit,
)
task4_outcome = await task4_kernel.turn(
    TurnRequest(
        session_key=INCIDENT["session_key"],
        interaction_id="task-4-notebook-advance",
        actor_user_id="U-sanitized",
        expected_revision=1,
        intent=Advance(),
    ),
    progress=NotebookProgress(),
)
task4_saved = await task4_repo.load_or_create(
    INCIDENT["session_key"], owner_user_id="U-sanitized"
)
task4_gym_assumptions = [
    assumption
    for assumption in task4_saved.assumptions
    if assumption.requirement_id == "skeleton.gym_placement"
]

assert isinstance(task4_outcome, AwaitingApproval)
assert task4_outcome.artifact.kind is ArtifactKind.SKELETON
assert len(task4_gym_assumptions) == 1
assert task4_planner.calls == 1
assert task4_commit.calls == 0
{
    "outcome": task4_outcome.kind,
    "target_artifact": task4_outcome.artifact.kind.value,
    "gym_assumption": task4_gym_assumptions[0].model_dump(mode="json"),
    "planner_calls": task4_planner.calls,
    "commit_calls": task4_commit.calls,
}


In [ ]:
# task-11-replay-driver: the deterministic replay of the 2026-08-29 incident.
# The fixture, the driver and the assertions all live in tests/replay/, and this
# cell imports the same code the suite runs. A notebook that reimplemented the
# replay would be a second answer to the same question, free to drift from the
# one that gates the branch.
import sys

REPO_ROOT = "../.."
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from tests.replay.test_timeboxing_incident_20260829 import (
    artifact_kinds,
    asked_requirement_ids,
    assumption_requirement_ids,
    load_fixture,
    replay_scenario,
    requirement_owners,
    unterminated_activities,
)

replay_fixture = load_fixture()
replay_fixture["recording"]

In [ ]:
# task-11-incident-evidence: AC1, AC3, AC4 and AC7 against the recorded turns.
from fateforger.agents.timeboxing.readiness import RequirementOwner
from fateforger.agents.timeboxing.session_contracts import ArtifactKind, DayType

incident_run = await replay_scenario("incident", replay_fixture)
incident_owners = requirement_owners(incident_run.snapshot)
incident_day = incident_run.snapshot.planning_day

# AC1 -- the day stays the day the host locked, whatever the calendar contained.
assert incident_day.date.isoformat() == "2026-08-29"
assert incident_day.iso_weekday == 6
assert incident_day.day_type is DayType.WEEKEND
# AC3 -- the delegated placements were decided by the planner and labelled.
assert "skeleton.gym_placement" in assumption_requirement_ids(incident_run.snapshot)
assert incident_owners["skeleton.gym_placement"] is RequirementOwner.PLANNER
assert incident_owners["skeleton.ordinary_placement"] is RequirementOwner.PLANNER
# ...and neither of them was ever put to the user as a question.
assert asked_requirement_ids(incident_run.outcomes) == {"skeleton.requested_activity"}
# AC4 -- the advance ends on a reviewable artifact, not another recap.
assert artifact_kinds(incident_run.outcomes)[-1] is ArtifactKind.SKELETON
assert incident_run.commit.calls == []

{
    "locked_day": incident_day.model_dump(mode="json"),
    "questions_asked": sorted(asked_requirement_ids(incident_run.outcomes)),
    "planner_assumptions": sorted(assumption_requirement_ids(incident_run.snapshot)),
    "artifacts_presented": [
        kind.value for kind in artifact_kinds(incident_run.outcomes)
    ],
    "commit_calls": incident_run.commit.calls,
    # Non-empty, and reported rather than asserted away: the kernel starts a
    # `resolving_context` progress activity and never terminates it, so the
    # Slack checklist keeps a spinner row after the turn ends. Locked as a
    # strict xfail in the replay suite; see task-11-report.md.
    "progress_activities_left_open": sorted(
        unterminated_activities(incident_run.progress.events)
    ),
}

In [ ]:
# task-11-edge-case-evidence: every recorded scenario and the outcome it ends on.
edge_case_evidence = {}
for scenario_name in replay_fixture["scenarios"]:
    run = await replay_scenario(scenario_name, replay_fixture)
    final = run.outcomes[-1]
    edge_case_evidence[scenario_name] = {
        "turns": len(run.outcomes),
        "final_outcome": final.kind,
        "failure_code": getattr(final, "code", None),
        "planner_calls": run.planner.calls,
        "questions_asked": sorted(asked_requirement_ids(run.outcomes)),
        "commit_calls": run.commit.calls,
    }

# The failure codes are minted by this system, so a route can branch on them
# without anyone reading a sentence.
assert (
    edge_case_evidence["planner_delegation_refused"]["failure_code"]
    == "illegal_user_blocker"
)
assert edge_case_evidence["stale_skeleton_approval"]["failure_code"] == "stale_approval"
assert (
    edge_case_evidence["superseded_harness_result"]["failure_code"]
    == "stale_session_revision"
)
assert (
    edge_case_evidence["planner_unavailable"]["failure_code"]
    == "dependency_unavailable"
)
assert edge_case_evidence["duplicate_delivery"]["planner_calls"] == 1
assert all(
    evidence["commit_calls"] == [] for evidence in edge_case_evidence.values()
)
edge_case_evidence

## Reviewer Checklist

- Does the design make artifacts—not transcript prose or a mutable stage variable—the authority?
- Can only a hard user-owned requirement cause a user question?
- Does every accepted Advance produce the target artifact in the same turn or a typed failure?
- Are date, candidate identity, persistence, and commit idempotency host-owned?
- Are Slack progress updates useful but observational and privacy-safe?
- Is Stage 3 presentation-only and Stage 4 the first patch boundary?


## Open Items

- **To decide:** final commit/push/merge authorization remains at handoff.
- **To do:** execute Tasks 7–13; run deterministic and live Slack replays; restart the chatbot for Hugo's test.
- **Blocked by:** draft PR publication remains deferred until commit/push authority; implementation is unblocked.


## Acceptance Criteria Checklist

- [ ] AC1 — planning date/day type remains locked and structurally confirmed
- [ ] AC2 — readiness is derived from downstream typed requirements
- [ ] AC3 — planner makes ordinary placement choices and labels assumptions
- [ ] AC4 — typed Advance produces the next reviewable artifact or typed failure
- [ ] AC5 — session survives restart without Slack transcript reconstruction
- [ ] AC6 — DeepSeek receives complete host-owned context and clean instructions
- [ ] AC7 — exact incident and edge cases replay successfully


## Implementation Evidence

Setup: isolated worktree at base `fd95a0e`; original dirty files absent; approved spec and plan present.

Task 2: missing-module RED observed; 10/10 contract tests GREEN; targeted Python 3.13 mypy passed; independent review clean after one fix round. Files: `session_contracts.py`, package exports, and `test_timeboxing_session_contracts.py`.

Task 3: missing-module RED observed; 8/8 readiness tests GREEN; combined Task 2+3 suite passed 18/18 with Ruff formatting/lint clean. The executable `incident-readiness-evidence` cell uses real typed contracts and asserts no hard user blocker plus planner-owned gym placement with `assume` resolution. Files: `readiness.py` and `test_timeboxing_readiness.py`.

Task 4: missing-module RED plus focused review regressions observed; 19/19 kernel tests and 37/37 cumulative Task 2–4 tests GREEN with Ruff and bounded mypy clean. The executable `task-4-kernel-evidence` cell uses the real kernel/repository/contracts and thin local port doubles to assert a same-turn `AwaitingApproval(SKELETON)`, one labeled gym-placement assumption, one planner call, and zero commit calls. Files: `adaptive_timeboxing.py` and `test_adaptive_timeboxing.py`.

Task 5: the SQLAlchemy/Alembic repository persists one versioned snapshot+outcome envelope, replays typed outcomes after restart, and rejects stale CAS writes. Direct greenlet runtime support on Python 3.11 arm64 and manual-only Alembic metadata are verified. 4/4 focused and 41/41 cumulative tests GREEN; disposable upgrade/downgrade/upgrade exact; independent re-review clean.

Task 6: schema-bound NL interpretation and structural Block Kit actions converge on typed intents. Strict v1 metadata fails closed; legacy defaults apply only without a version. UI action envelopes preserve session/revision for one executor decode. 17/17 focused and 61/61 adjacent tests GREEN; independent re-review clean.

Task 11: the 2026-08-29 conversation replays deterministically against recorded typed inputs. `tests/replay/fixtures/timeboxing_incident_20260829.json` carries the sanitized recording -- no tokens, no raw reasoning, no calendar event identifiers -- and every payload in it is parsed by the production contracts, whose `extra="forbid"` is what stops a stray payload riding along. `tests/replay/test_timeboxing_incident_20260829.py` replays eight scenarios: the incident itself plus a hard user-owned conflict, a refused planner delegation, a date relock after the skeleton exists, a duplicate Slack delivery, a stale skeleton approval, a superseded harness result, and an unreachable planner. 14 passed and 2 strict xfails; 1920 passed / 10 skipped / 3 xfailed across `tests/unit tests/integration tests/replay`. "No outcome asks for gym time" is expressed over requirement IDs and their catalog owners, never over question prose. The two strict xfails are reported production defects in `AdaptiveTimeboxing` progress emission, not weakened assertions: `resolving_context` starts and never terminates, and `planning` reports `succeeded` on a turn that failed. Both are written up in `.superpowers/sdd/2026-08-29-adaptive-timeboxing-session-kernel/task-11-report.md`.


## Extraction Map (Notebook -> Artifacts)

| Notebook material | Durable destination |
|---|---|
| Contracts/readiness experiments | `src/fateforger/agents/timeboxing/` + unit tests |
| Persistence experiments | Slack store adapter + Alembic migration + tests |
| Harness/result experiments | Slack harness adapters + profile docs + tests |
| Incident assertions | replay fixture/tests |
| Architecture and runbook conclusions | nearest READMEs/docs |


## Closeout / Remaining Notebook-Only Content

Keep only the minimal replay/workbench cells, live integration recipes, and decision evidence after extraction. Production logic and deterministic assertions must live in `src/`, `tests/`, and durable docs before this notebook can move to `Extraction complete`.
